1. 파일 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# 데이터 경로
paths = [
    Path("../../data/cal01_base_v1.csv"),
    Path("../data/cal01_base_v1.csv"),
    Path("data/cal01_base_v1.csv"),
    Path("cal01_base_v1.csv")
]

CSV_PATH = next((p for p in paths if p.exists()), None)

if CSV_PATH is None:
    raise FileNotFoundError("cal01_base_v1.csv 파일을 찾을 수 없습니다.")

# 분석 설정
Z_THRESHOLD = 3.0
THRESHOLDS = [2.0, 2.5, 3.0, 3.5]

# 센서 21개
TC_SENSORS = [f"TC-Z{i}" for i in range(1, 10)]
TC_SENSORS += ["TC-OUT1", "TC-OUT2"]

OP_SENSORS = [f"OP-Z{i}" for i in range(1, 9)]

SENSORS = ["CLN-IN"] + TC_SENSORS + OP_SENSORS + ["CP-Z4"]

# 필요한 컬럼만 읽기
META = [
    "ROW_ID", "LOT_NO", "MEAS_DT", "GRADE",
    "LOT_ELAPSED_MIN", "EXCLUDED", "LABEL"
]

df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8-sig",
    usecols=META + SENSORS,
    dtype={"LOT_NO": str}
)

df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])

df["EXCLUDED"] = (
    df["EXCLUDED"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1"])
)

for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce")

df = df.sort_values(["MEAS_DT", "ROW_ID"]).reset_index(drop=True)

print("전체 행 수:", len(df))
print("LOT 개수:", df["LOT_NO"].nunique())
print("센서 개수:", len(SENSORS))
print("분석 제외 행:", df["EXCLUDED"].sum())

display(df.head())

2. LOT별 평균·표준편차 계산

In [ ]:
# 제외 대상은 계산에 사용하지 않음
valid = df.loc[~df["EXCLUDED"]].copy()

# LOT별 센서 평균
lot_means = valid.groupby("LOT_NO")[SENSORS].mean()

# LOT별 센서 표준편차
lot_stds = valid.groupby("LOT_NO")[SENSORS].std(ddof=0)

# LOT별 측정 개수
lot_counts = valid.groupby("LOT_NO")[SENSORS].count()

# 결과 정리
stats_list = []

for lot in lot_means.index:
    for sensor in SENSORS:
        stats_list.append({
            "LOT_NO": lot,
            "SENSOR": sensor,
            "MEAN": lot_means.loc[lot, sensor],
            "STD": lot_stds.loc[lot, sensor],
            "MEASURED_N": lot_counts.loc[lot, sensor]
        })

lot_stats = pd.DataFrame(stats_list)

display(lot_stats.head(30))

3. 14개 LOT의 Z-score 계산

In [ ]:
Z_COLS = []

for sensor in SENSORS:

    z_col = f"Z_{sensor}"
    Z_COLS.append(z_col)

    # LOT별 평균과 표준편차 연결
    mean = valid["LOT_NO"].map(lot_means[sensor])
    std = valid["LOT_NO"].map(lot_stds[sensor])

    # 표준편차가 0이면 계산하지 않음
    std = std.replace(0, np.nan)

    # Z-score 계산
    df[z_col] = np.nan

    df.loc[valid.index, z_col] = (
        (valid[sensor] - mean) / std
    ).to_numpy()

print("Z-score 계산 완료")

display(
    df[
        ["LOT_NO", "MEAS_DT", "TC-Z3", "Z_TC-Z3",
         "OP-Z3", "Z_OP-Z3", "EXCLUDED"]
    ].head(20)
)

4. 임계값별 이상 후보 비교

In [ ]:
threshold_list = []

for lot, group in df.groupby("LOT_NO"):

    for sensor in SENSORS:

        z = group[f"Z_{sensor}"].dropna().abs()

        for threshold in THRESHOLDS:

            candidate_n = (z >= threshold).sum()

            threshold_list.append({
                "LOT_NO": lot,
                "SENSOR": sensor,
                "THRESHOLD": threshold,
                "MEASURED_N": len(z),
                "CANDIDATE_N": candidate_n,
                "CANDIDATE_PCT": (
                    candidate_n / len(z) * 100
                    if len(z) > 0 else np.nan
                )
            })

threshold_summary = pd.DataFrame(threshold_list)

display(threshold_summary.head(30))

5. 실제 이상 후보 추출

In [ ]:
candidate_list = []
summary_list = []

for lot, group in df.groupby("LOT_NO"):

    for sensor in SENSORS:

        z_col = f"Z_{sensor}"

        # 이상 후보
        mask = group[z_col].abs() >= Z_THRESHOLD

        # 센서별 요약
        measured_n = group[z_col].notna().sum()
        candidate_n = mask.sum()

        summary_list.append({
            "LOT_NO": lot,
            "SENSOR": sensor,
            "MEASURED_N": measured_n,
            "CANDIDATE_N": candidate_n,
            "CANDIDATE_PCT": (
                candidate_n / measured_n * 100
                if measured_n > 0 else np.nan
            )
        })

        # 이상 후보가 있을 때만 추출
        if candidate_n == 0:
            continue

        temp = group.loc[
            mask,
            [
                "ROW_ID", "LOT_NO", "MEAS_DT",
                "GRADE", "LOT_ELAPSED_MIN",
                "EXCLUDED", "LABEL"
            ]
        ].copy()

        temp["SENSOR"] = sensor
        temp["ACTUAL"] = group.loc[mask, sensor].values
        temp["Z_SCORE"] = group.loc[mask, z_col].values
        temp["ABS_Z"] = temp["Z_SCORE"].abs()

        candidate_list.append(temp)

lot_sensor_summary = pd.DataFrame(summary_list)

candidate_columns = [
    "ROW_ID", "LOT_NO", "MEAS_DT", "GRADE",
    "LOT_ELAPSED_MIN", "EXCLUDED", "LABEL",
    "SENSOR", "ACTUAL", "Z_SCORE", "ABS_Z"
]

if candidate_list:
    candidates = pd.concat(candidate_list, ignore_index=True)
    candidates = candidates.sort_values(
        ["LOT_NO", "SENSOR", "MEAS_DT"]
    ).reset_index(drop=True)
else:
    candidates = pd.DataFrame(columns=candidate_columns)

print("이상 후보 건수:", len(candidates))

display(lot_sensor_summary.head(30))
display(candidates.head(30))

6. 이상 후보의 연속 지속시간

In [ ]:
runs_list = []

for (lot, sensor), group in candidates.groupby(
    ["LOT_NO", "SENSOR"]
):

    group = group.sort_values("MEAS_DT").copy()

    # 1초 간격이 끊기면 새로운 구간
    group["RUN_ID"] = (
        group["MEAS_DT"]
        .diff()
        .dt.total_seconds()
        .ne(1)
        .cumsum()
    )

    runs = group.groupby("RUN_ID").agg(
        START=("MEAS_DT", "min"),
        END=("MEAS_DT", "max"),
        DURATION_SEC=("MEAS_DT", "size"),
        MAX_ABS_Z=("ABS_Z", "max")
    ).reset_index(drop=True)

    runs["LOT_NO"] = lot
    runs["SENSOR"] = sensor

    runs_list.append(runs)

run_columns = [
    "LOT_NO", "SENSOR", "START", "END",
    "DURATION_SEC", "MAX_ABS_Z"
]

if runs_list:
    runs_summary = pd.concat(runs_list, ignore_index=True)
    runs_summary = runs_summary[run_columns]
    runs_summary = runs_summary.sort_values(
        "DURATION_SEC", ascending=False
    ).reset_index(drop=True)
else:
    runs_summary = pd.DataFrame(columns=run_columns)

display(runs_summary.head(30))

7. LOT별 이상 후보 시각화

In [ ]:
lot_summary = lot_sensor_summary.groupby("LOT_NO").agg(
    MEASURED_N=("MEASURED_N", "sum"),
    CANDIDATE_N=("CANDIDATE_N", "sum")
)

lot_summary["CANDIDATE_PCT"] = (
    lot_summary["CANDIDATE_N"]
    / lot_summary["MEASURED_N"] * 100
)

ax = lot_summary["CANDIDATE_PCT"].plot(
    kind="bar",
    figsize=(12, 5)
)

ax.set_title("LOT별 Z-score 이상 후보 비율")
ax.set_xlabel("LOT_NO")
ax.set_ylabel("Candidate (%)")

for bar in ax.patches:
    value = bar.get_height()

    ax.annotate(
        f"{value:.2f}%",
        (bar.get_x() + bar.get_width() / 2, value),
        ha="center",
        va="bottom",
        fontsize=9,
        xytext=(0, 3),
        textcoords="offset points"
    )

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

8. CSV 파일 저장

In [ ]:
REPORT_DIR = CSV_PATH.parent.parent / "data" / "sayletskin" /"zscore_all_lots"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 전체 행과 21개 Z-score
score_csv = df[
    META + Z_COLS
].copy()

outputs = {
    "01_zscore_lot_stats.csv": lot_stats,
    "02_zscore_thresholds.csv": threshold_summary,
    "03_zscore_lot_sensor_summary.csv": lot_sensor_summary,
    "04_zscore_candidates.csv": candidates,
    "05_zscore_runs.csv": runs_summary,
    "06_zscore_all_rows.csv": score_csv
}

for filename, result in outputs.items():

    result.to_csv(
        REPORT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

    print("저장 완료:", filename)

print("\n전체 저장 완료:", REPORT_DIR)

그래프 및 저장

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from pathlib import Path
# from matplotlib import font_manager, rcParams
# import zipfile

# # --------------------------------------------------
# # 1. 한글 폰트 설정
# # --------------------------------------------------
# installed = {f.name for f in font_manager.fontManager.ttflist}

# for cand in [
#     "Malgun Gothic",
#     "AppleGothic",
#     "NanumGothic",
#     "NanumBarunGothic",
#     "Noto Sans CJK KR"
# ]:
#     if cand in installed:
#         rcParams["font.family"] = cand
#         break

# rcParams["axes.unicode_minus"] = False

# # --------------------------------------------------
# # 2. 파일 경로
# # --------------------------------------------------

# # CSV가 저장된 폴더
# BASE_DIR = Path("../../data/sayletskin/zscore_all_lots")

# THRESHOLDS_CSV = BASE_DIR / "02_zscore_thresholds.csv"
# LOT_SENSOR_CSV = BASE_DIR / "03_zscore_lot_sensor_summary.csv"
# CANDIDATES_CSV = BASE_DIR / "04_zscore_candidates.csv"
# RUNS_CSV = BASE_DIR / "05_zscore_runs.csv"

# # 그래프 저장 폴더
# OUT_DIR = Path("../../figures/sayletskin/zscore_all_lots")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# # --------------------------------------------------
# # 3. CSV 읽기
# # --------------------------------------------------

# thresholds = pd.read_csv(THRESHOLDS_CSV, dtype={"LOT_NO": str})
# lot_sensor = pd.read_csv(LOT_SENSOR_CSV, dtype={"LOT_NO": str})
# candidates = pd.read_csv(CANDIDATES_CSV, dtype={"LOT_NO": str})
# runs = pd.read_csv(RUNS_CSV, dtype={"LOT_NO": str})

# print("CSV 파일 불러오기 완료")

# # --------------------------------------------------
# # 4. LOT별 색상 만들기
# # --------------------------------------------------
# lots = sorted(lot_sensor["LOT_NO"].dropna().astype(str).unique().tolist())

# # matplotlib 기본 colormap 사용
# cmap = plt.get_cmap("tab20")

# lot_colors = {}
# for i, lot in enumerate(lots):
#     lot_colors[lot] = cmap(i % 20)

# # --------------------------------------------------
# # 5. 막대 위/옆에 수치 표시 함수
# # --------------------------------------------------
# def add_bar_labels(ax, percent=False, horizontal=False, suffix=""):
#     if horizontal:
#         xmax = ax.get_xlim()[1]
#         offset = xmax * 0.01

#         for p in ax.patches:
#             value = p.get_width()
#             y = p.get_y() + p.get_height() / 2

#             if percent:
#                 text = f"{value:.2f}%"
#             else:
#                 text = f"{value:.0f}{suffix}"

#             ax.text(
#                 value + offset,
#                 y,
#                 text,
#                 va="center",
#                 ha="left",
#                 fontsize=9
#             )
#     else:
#         ymax = ax.get_ylim()[1]
#         offset = ymax * 0.01

#         for p in ax.patches:
#             value = p.get_height()
#             x = p.get_x() + p.get_width() / 2

#             if percent:
#                 text = f"{value:.2f}%"
#             else:
#                 text = f"{value:.0f}{suffix}"

#             ax.text(
#                 x,
#                 value + offset,
#                 text,
#                 va="bottom",
#                 ha="center",
#                 fontsize=9
#             )

# saved_files = []

# # --------------------------------------------------
# # 6. 그래프 1
# # LOT별 전체 이상 후보 비율
# # --------------------------------------------------
# lot_pct = lot_sensor.groupby("LOT_NO", as_index=False)[["MEASURED_N", "CANDIDATE_N"]].sum()
# lot_pct["CANDIDATE_PCT"] = lot_pct["CANDIDATE_N"] / lot_pct["MEASURED_N"] * 100
# lot_pct = lot_pct.sort_values("LOT_NO")

# plt.figure(figsize=(12, 6))
# ax = plt.gca()

# ax.bar(
#     lot_pct["LOT_NO"],
#     lot_pct["CANDIDATE_PCT"],
#     color=[lot_colors[lot] for lot in lot_pct["LOT_NO"]]
# )

# ax.set_title("LOT별 전체 이상 후보 비율")
# ax.set_xlabel("LOT_NO")
# ax.set_ylabel("이상 후보 비율 (%)")
# plt.xticks(rotation=45)

# add_bar_labels(ax, percent=True)

# plt.tight_layout()

# file_1 = OUT_DIR / "01_LOT별_전체_이상후보_비율.png"
# plt.savefig(file_1, dpi=150, bbox_inches="tight")
# plt.show()
# plt.close()

# saved_files.append(file_1)

# # --------------------------------------------------
# # 7. 그래프 2
# # LOT별 전체 이상 후보 건수
# # --------------------------------------------------
# lot_cnt = candidates.groupby("LOT_NO", as_index=False).size()
# lot_cnt = lot_cnt.rename(columns={"size": "CANDIDATE_COUNT"})
# lot_cnt = lot_cnt.sort_values("LOT_NO")

# plt.figure(figsize=(12, 6))
# ax = plt.gca()

# ax.bar(
#     lot_cnt["LOT_NO"],
#     lot_cnt["CANDIDATE_COUNT"],
#     color=[lot_colors[lot] for lot in lot_cnt["LOT_NO"]]
# )

# ax.set_title("LOT별 전체 이상 후보 건수")
# ax.set_xlabel("LOT_NO")
# ax.set_ylabel("이상 후보 건수")
# plt.xticks(rotation=45)

# add_bar_labels(ax)

# plt.tight_layout()

# file_2 = OUT_DIR / "02_LOT별_전체_이상후보_건수.png"
# plt.savefig(file_2, dpi=150, bbox_inches="tight")
# plt.show()
# plt.close()

# saved_files.append(file_2)

# # --------------------------------------------------
# # 8. 그래프 3
# # 임계값별 전체 이상 후보 비율
# # --------------------------------------------------
# thr = thresholds.groupby("THRESHOLD", as_index=False)[["MEASURED_N", "CANDIDATE_N"]].sum()
# thr["CANDIDATE_PCT"] = thr["CANDIDATE_N"] / thr["MEASURED_N"] * 100
# thr = thr.sort_values("THRESHOLD")

# plt.figure(figsize=(10, 6))
# ax = plt.gca()

# ax.bar(
#     thr["THRESHOLD"].astype(str),
#     thr["CANDIDATE_PCT"]
# )

# ax.set_title("임계값별 전체 이상 후보 비율")
# ax.set_xlabel("Z-score 임계값")
# ax.set_ylabel("이상 후보 비율 (%)")

# add_bar_labels(ax, percent=True)

# plt.tight_layout()

# file_3 = OUT_DIR / "03_임계값별_전체_이상후보_비율.png"
# plt.savefig(file_3, dpi=150, bbox_inches="tight")
# plt.show()
# plt.close()

# saved_files.append(file_3)

# # --------------------------------------------------
# # 9. 그래프 4
# # LOT별 최장 연속 이상 지속시간
# # --------------------------------------------------
# lot_max_run = runs.groupby("LOT_NO", as_index=False)["DURATION_SEC"].max()
# lot_max_run = lot_max_run.sort_values("LOT_NO")

# plt.figure(figsize=(12, 6))
# ax = plt.gca()

# ax.bar(
#     lot_max_run["LOT_NO"],
#     lot_max_run["DURATION_SEC"],
#     color=[lot_colors[lot] for lot in lot_max_run["LOT_NO"]]
# )

# ax.set_title("LOT별 최장 연속 이상 지속시간")
# ax.set_xlabel("LOT_NO")
# ax.set_ylabel("최장 지속시간 (초)")
# plt.xticks(rotation=45)

# add_bar_labels(ax, suffix="초")

# plt.tight_layout()

# file_4 = OUT_DIR / "04_LOT별_최장_연속이상_지속시간.png"
# plt.savefig(file_4, dpi=150, bbox_inches="tight")
# plt.show()
# plt.close()

# saved_files.append(file_4)

# # --------------------------------------------------
# # 10. 그래프 5
# # LOT별 연속 이상 구간 수
# # --------------------------------------------------
# lot_run_cnt = runs.groupby("LOT_NO", as_index=False).size()
# lot_run_cnt = lot_run_cnt.rename(columns={"size": "RUN_COUNT"})
# lot_run_cnt = lot_run_cnt.sort_values("LOT_NO")

# plt.figure(figsize=(12, 6))
# ax = plt.gca()

# ax.bar(
#     lot_run_cnt["LOT_NO"],
#     lot_run_cnt["RUN_COUNT"],
#     color=[lot_colors[lot] for lot in lot_run_cnt["LOT_NO"]]
# )

# ax.set_title("LOT별 연속 이상 구간 수")
# ax.set_xlabel("LOT_NO")
# ax.set_ylabel("연속 이상 구간 수")
# plt.xticks(rotation=45)

# add_bar_labels(ax)

# plt.tight_layout()

# file_5 = OUT_DIR / "05_LOT별_연속이상_구간수.png"
# plt.savefig(file_5, dpi=150, bbox_inches="tight")
# plt.show()
# plt.close()

# saved_files.append(file_5)

# # --------------------------------------------------
# # 11. 그래프 6
# # LOT별 대표 센서 이상 후보 비율
# # 각 LOT에서 이상 후보 비율이 가장 높은 센서 1개만 표시
# # --------------------------------------------------
# top_sensor = lot_sensor.sort_values(
#     ["LOT_NO", "CANDIDATE_PCT"],
#     ascending=[True, False]
# ).groupby("LOT_NO").head(1).copy()

# top_sensor = top_sensor.sort_values("LOT_NO")
# top_sensor["LOT_SENSOR"] = top_sensor["LOT_NO"] + "\n(" + top_sensor["SENSOR"] + ")"

# plt.figure(figsize=(13, 6))
# ax = plt.gca()

# ax.bar(
#     top_sensor["LOT_SENSOR"],
#     top_sensor["CANDIDATE_PCT"],
#     color=[lot_colors[lot] for lot in top_sensor["LOT_NO"]]
# )

# ax.set_title("LOT별 대표 센서 이상 후보 비율")
# ax.set_xlabel("LOT_NO (대표 센서)")
# ax.set_ylabel("이상 후보 비율 (%)")
# plt.xticks(rotation=45, ha="right")

# add_bar_labels(ax, percent=True)

# plt.tight_layout()

# file_6 = OUT_DIR / "06_LOT별_대표센서_이상후보_비율.png"
# plt.savefig(file_6, dpi=150, bbox_inches="tight")
# plt.show()
# plt.close()

# saved_files.append(file_6)

# # --------------------------------------------------
# # 12. ZIP 파일 생성
# # --------------------------------------------------
# zip_path = OUT_DIR / "zscore_all_lots_charts.zip"

# with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for file_path in saved_files:
#         zf.write(file_path, arcname=file_path.name)

# print("그래프 저장 완료:")
# for file_path in saved_files:
#     print(file_path)

# print("\nZIP 파일:")
# print(zip_path)

OP-Z3 분석

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

BASE_DIR = Path("../../data/sayletskin/zscore_all_lots")

# 실제 파일명 확인
csv_path = BASE_DIR / "03_zscore_lot_sensor_summary.csv"

if not csv_path.exists():
    csv_path = BASE_DIR / "03_zscore_lot_sensor_summary(1).csv"

summary = pd.read_csv(csv_path, dtype={"LOT_NO": str})

# OP-Z3만 추출
op3 = summary[summary["SENSOR"] == "OP-Z3"].copy()
op3 = op3.sort_values("LOT_NO")

display(op3)

# LOT별 색상
colors = plt.cm.tab20(range(len(op3)))

fig, ax = plt.subplots(figsize=(14, 5))

bars = ax.bar(
    op3["LOT_NO"],
    op3["CANDIDATE_PCT"],
    color=colors
)

ax.bar_label(bars, fmt="%.2f%%", padding=3)

ax.set_title("OP-Z3 LOT별 Z-score 이상 후보 비율")
ax.set_xlabel("LOT_NO")
ax.set_ylabel("이상 후보 비율 (%)")
ax.set_ylim(0, op3["CANDIDATE_PCT"].max() * 1.2)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

BASE_DIR = Path("../../data/sayletskin/zscore_all_lots")

# 실제 파일명 확인
csv_path = BASE_DIR / "03_zscore_lot_sensor_summary.csv"

if not csv_path.exists():
    csv_path = BASE_DIR / "03_zscore_lot_sensor_summary(1).csv"

summary = pd.read_csv(csv_path, dtype={"LOT_NO": str})

# OP-Z3만 추출
op3 = summary[summary["SENSOR"] == "OP-Z3"].copy()
op3 = op3.sort_values("LOT_NO")

display(op3)

# LOT별 색상
colors = plt.cm.tab20(range(len(op3)))

fig, ax = plt.subplots(figsize=(14, 5))

bars = ax.bar(
    op3["LOT_NO"],
    op3["CANDIDATE_PCT"],
    color=colors
)

ax.bar_label(bars, fmt="%.2f%%", padding=3)

ax.set_title("OP-Z3 LOT별 Z-score 이상 후보 비율")
ax.set_xlabel("LOT_NO")
ax.set_ylabel("이상 후보 비율 (%)")
ax.set_ylim(0, op3["CANDIDATE_PCT"].max() * 1.2)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

CP-Z4 LOT별 이상 후보 비율

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

BASE_DIR = Path("../../data/sayletskin/zscore_all_lots")

csv_path = BASE_DIR / "03_zscore_lot_sensor_summary.csv"

if not csv_path.exists():
    csv_path = BASE_DIR / "03_zscore_lot_sensor_summary(1).csv"

summary = pd.read_csv(csv_path, dtype={"LOT_NO": str})

# CP-Z4만 추출
cp = summary[summary["SENSOR"] == "CP-Z4"].copy()
cp = cp.sort_values("LOT_NO")

display(cp)

# LOT별 다른 색상
colors = plt.cm.tab20(range(len(cp)))

fig, ax = plt.subplots(figsize=(14, 5))

bars = ax.bar(
    cp["LOT_NO"],
    cp["CANDIDATE_PCT"],
    color=colors
)

ax.bar_label(bars, fmt="%.2f%%", padding=3)

ax.set_title("CP-Z4 LOT별 Z-score 이상 후보 비율")
ax.set_xlabel("LOT_NO")
ax.set_ylabel("이상 후보 비율 (%)")
ax.set_ylim(0, max(cp["CANDIDATE_PCT"].max() * 1.2, 0.1))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# CSV 경로
CSV_PATH = Path(
    "../../data/sayletskin/zscore_all_lots/06_zscore_all_rows.csv"
)

df = pd.read_csv(
    CSV_PATH,
    usecols=["LOT_NO", "MEAS_DT", "Z_CP-Z4"],
    dtype={"LOT_NO": str},
    parse_dates=["MEAS_DT"]
)

# LOT 240058만 선택
lot = df[df["LOT_NO"] == "240058"].copy()
lot = lot.sort_values("MEAS_DT")

# 이상 후보
mask = lot["Z_CP-Z4"].abs() >= 3
abnormal = lot.loc[mask]

# 가장 크게 튄 시점
peak_idx = lot["Z_CP-Z4"].abs().idxmax()
peak_time = lot.loc[peak_idx, "MEAS_DT"]

# 전후 10분
zoom = lot[
    lot["MEAS_DT"].between(
        peak_time - pd.Timedelta(minutes=10),
        peak_time + pd.Timedelta(minutes=10)
    )
]

fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# ① 전체 구간
axes[0].plot(
    lot["MEAS_DT"], lot["Z_CP-Z4"],
    color="steelblue", linewidth=1
)

axes[0].scatter(
    abnormal["MEAS_DT"], abnormal["Z_CP-Z4"],
    color="red", s=12, label="이상 후보"
)

axes[0].set_title("LOT 240058 - CP-Z4 전체 Z-score")
axes[0].legend()

# ② 튄 부분 확대
axes[1].plot(
    zoom["MEAS_DT"], zoom["Z_CP-Z4"],
    color="steelblue", linewidth=1
)

zoom_abnormal = zoom[zoom["Z_CP-Z4"].abs() >= 3]

axes[1].scatter(
    zoom_abnormal["MEAS_DT"],
    zoom_abnormal["Z_CP-Z4"],
    color="red", s=20, label="이상 후보"
)

axes[1].set_title("CP-Z4 최대 이상 시점 ±10분")
axes[1].legend()

#